# TransformerQEC: Evaluation & Comparison

Evaluate the transformer decoder against Minimum Weight Perfect Matching (MWPM) across code distances and physical error rates. Produce logical error rate curves and estimate the error correction threshold.

**Pre-requisite:** Run `02_model_and_training.ipynb` for each distance currently in scope (d=3, 5) to produce timestamped checkpoint files (`best_d{d}_{YYYYMMDD_HHMMSS}_checkpoint.pkl`). The eval loader picks the most recent timestamp per distance. d=7 is trainable once the d=5 DIPE + OTF cycle validates.

In [ ]:
# Install pinned deps from repo requirements.txt (Drive-mounted path).
# If requirements.txt isn't reachable yet (fresh Colab), fall back to inline pins.
import os
_REQ_PATH = '/content/drive/MyDrive/TransformerQEC/requirements.txt'
if os.path.exists(_REQ_PATH):
    !pip install -q -r {_REQ_PATH}
else:
    !pip install -q "stim>=1.14,<2" "pymatching>=2,<3" "flax>=0.8" "numpy<2"

In [2]:
import gc
import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
import stim
import pymatching
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

print(f'JAX backend: {jax.default_backend()}')

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX backend: tpu


## Model Definition & Data Utilities

Model architecture must match the one used during training to load checkpoint weights correctly.

In [3]:
import sys
from google.colab import drive

# 1. Mount your Google Drive (this will ask for permission pop-ups)
drive.mount('/content/drive')

# 2. Tell Python where to look for model.py
# CRITICAL: You must update this path to match exactly where you uploaded the folder!
FOLDER_PATH = '/content/drive/MyDrive/TransformerQEC/notebooks' 

if FOLDER_PATH not in sys.path:
    sys.path.append(FOLDER_PATH)

# 4. Now the import will work perfectly
from model import TransformerQEC

Mounted at /content/drive


In [ ]:
# --- Data utilities ---

def make_circuit(d, p, rounds=None):
    if rounds is None:
        rounds = d
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=d, rounds=rounds,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p)


def sample_syndromes(circuit, num_shots):
    sampler = circuit.compile_detector_sampler()
    det, obs = sampler.sample(num_shots, separate_observables=True)
    return det.astype(np.float32), obs[:, 0].astype(np.int64)


def get_detector_coords(d, rounds=None):
    """Extract raw integer detector (x, y, round) coordinates (DIPE).

    Coordinates are origin-shifted per-axis so each axis starts at 0,
    but NOT normalized or rescaled — the same physical lattice cell
    yields the same coordinate value across distances, which is what
    makes the (DIPE-form) RoPE distance-invariant.
    """
    circuit = make_circuit(d, p=0.01, rounds=rounds)
    raw = circuit.get_detector_coordinates()
    num_det = circuit.num_detectors
    coords = np.zeros((num_det, 3), dtype=np.float32)
    for det_idx, c in raw.items():
        coords[det_idx] = c[:3]
    for axis in range(3):
        coords[:, axis] = coords[:, axis] - coords[:, axis].min()
    return coords

## MWPM Baseline

In [5]:
def mwpm_decode(circuit, syndromes_bool):
    """Decode using Minimum Weight Perfect Matching via PyMatching."""
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    predictions = matcher.decode_batch(syndromes_bool)
    return predictions[:, 0]

## Load Trained Models

Load pre-trained checkpoints for each code distance from disk. These `.pkl` files are produced by `02_model_and_training.ipynb`.

In [ ]:
# Mount Google Drive and list available timestamped checkpoints.
import re
from pathlib import Path
CKPT_DIR = Path('/content/drive/My Drive/TransformerQEC/results')
assert CKPT_DIR.exists(), f"Checkpoint directory not found: {CKPT_DIR}"
print(f"Checkpoint dir: {CKPT_DIR}")
_STAMP_RE = re.compile(r'best_d(\d+)_(\d{8}_\d{6})_checkpoint\.pkl$')
_candidates_by_d = {}
for f in sorted(CKPT_DIR.glob('best_d*_*_checkpoint.pkl')):
    m = _STAMP_RE.search(f.name)
    if not m:
        continue
    d_val, stamp = int(m.group(1)), m.group(2)
    _candidates_by_d.setdefault(d_val, []).append((stamp, f))
    print(f"  d={d_val} stamp={stamp}  {f.name} ({f.stat().st_size:,} bytes)")

In [ ]:
# Accept both the bulk-materialized Cycle A/B/C training regime and the
# on-the-fly Cycle D regime; both share the same model math + RoPE angles.
EXPECTED_MODEL_VERSIONS = {'dipe-no-mask', 'dipe-no-mask-otf'}

#DISTANCES = [3, 5, 7]
DISTANCES = [3, 5]
#DISTANCES = [3]
# Optional per-distance timestamp pin — set e.g. {3: '20260417_093012'} to
# override the "latest" selection. Leave empty to always pick the newest.
CHECKPOINT_OVERRIDES = {}

models = {}

for d in DISTANCES:
    jax.clear_caches()
    gc.collect()

    candidates = _candidates_by_d.get(d, [])
    if not candidates:
        print(f'd={d}: no checkpoint found in {CKPT_DIR}, skipping')
        continue

    # Pick override if specified, else latest by timestamp (ISO sort).
    pin = CHECKPOINT_OVERRIDES.get(d)
    if pin is not None:
        match = [(s, p) for s, p in candidates if s == pin]
        if not match:
            print(f'd={d}: pinned stamp {pin!r} not found, skipping')
            continue
        stamp, path = match[0]
    else:
        candidates_sorted = sorted(candidates, key=lambda sp: sp[0])
        stamp, path = candidates_sorted[-1]
        if len(candidates) > 1:
            print(f'd={d}: {len(candidates)} checkpoints available, using latest ({stamp})')

    with open(path, 'rb') as f:
        ckpt = pickle.load(f)

    # Reject legacy checkpoints — RoPE math changed under DIPE; old Q/K
    # projections are no longer aligned with the new positional angles.
    ckpt_version = ckpt.get('config', {}).get('model_version')
    if ckpt_version not in EXPECTED_MODEL_VERSIONS:
        print(f'd={d}: SKIPPED — checkpoint model_version={ckpt_version!r} '
              f'(expected one of {sorted(EXPECTED_MODEL_VERSIONS)!r}). '
              f'Retrain with current model.py.')
        continue

    cfg = ckpt['config']

    # Override d_model from actual param shape (guards against wrong config)
    cfg['d_model'] = int(ckpt['params']['Dense_0']['kernel'].shape[-1])

    # Load coords from checkpoint, or recompute if missing (old checkpoint)
    if 'coords' in ckpt:
        coords = ckpt['coords']
    else:
        coords = get_detector_coords(d)
    seq_len = coords.shape[0]

    model = TransformerQEC(
        d_model=cfg['d_model'],
        num_heads=cfg.get('num_heads', 4),
        num_layers=cfg.get('num_layers', 4),
        ffn_dim=cfg.get('ffn_dim', 2048),
        code_distance=d,
        measurement_rounds=d)
    models[d] = {
        'params': ckpt['params'],
        'seq_len': seq_len,
        'model': model,
        'coords': coords,
        'timestamp': stamp,
        'path': str(path),
        'version': ckpt_version,
    }
    print(f'd={d}: loaded stamp={stamp} version={ckpt_version!r} '
          f'(seq_len={seq_len}, coords={coords.shape}, '
          f'd_model={model.d_model})')

print(f'Loaded models for distances: {list(models.keys())}')

## Evaluation Sweep

Evaluate both decoders on fresh test data across all (distance, error rate) pairs.

In [ ]:
def wilson_ci(successes, total, z=1.96):
    """Wilson 95% CI for a Bernoulli proportion.

    Better than Wald for small counts (doesn't go negative), cheaper than
    Clopper-Pearson. Returns (lo, hi) clipped to [0, 1].
    """
    if total == 0:
        return 0.0, 1.0
    p_hat = successes / total
    denom = 1.0 + z * z / total
    center = (p_hat + z * z / (2.0 * total)) / denom
    spread = z * np.sqrt(p_hat * (1.0 - p_hat) / total +
                         z * z / (4.0 * total * total)) / denom
    return max(0.0, center - spread), min(1.0, center + spread)


NUM_P = 10
P_EVAL = np.geomspace(0.001, 0.01, NUM_P).tolist()
NUM_TEST = 100_000

results = {}  # (d, p) -> {mwpm_ler, transformer_ler, *_ci_lo, *_ci_hi}

for d in DISTANCES:
    jax.clear_caches()
    gc.collect()

    info = models[d]
    model = info['model']
    params = jax.device_put(info['params'])  # ensure on device
    seq_len = info['seq_len']
    coords_d = jax.device_put(info['coords'])  # (seq_len, 3)   
    eval_batch = 128

    # Pad NUM_TEST to exact multiple of eval_batch for uniform scan shapes
    n_padded = ((NUM_TEST + eval_batch - 1) // eval_batch) * eval_batch
    pad_n = n_padded - NUM_TEST
    n_eval_batches = n_padded // eval_batch

    # Build scan-based inference (one compiled XLA program per distance,
    # reused across all p values since shapes are identical).
    @jax.jit
    def predict_all(params, syn_batched, p_batched, lab_batched, mask_batched):
        """Count correct predictions via scan. Mask excludes padding."""
        def body(correct, batch):
            syn, pe, lab, mask = batch
            preds = model.apply({'params': params}, syn, pe, coords_d).argmax(-1)
            correct = correct + jnp.where(mask, preds == lab, False).sum()
            return correct, None
        correct, _ = jax.lax.scan(
            body, jnp.int32(0),
            (syn_batched, p_batched, lab_batched, mask_batched))
        return correct

    print(f'd={d}: eval_batch={eval_batch}, '
          f'{n_eval_batches} scan iters, {pad_n} padded')

    for p in P_EVAL:
        circuit = make_circuit(d, p)
        syndromes, labels = sample_syndromes(circuit, NUM_TEST)

        # ---- MWPM (C++ baseline, cannot be JIT'd) ----
        mwpm_preds = mwpm_decode(circuit, syndromes.astype(np.bool_))
        mwpm_failures = int((mwpm_preds != labels).sum())
        mwpm_ler = mwpm_failures / NUM_TEST
        mwpm_ci_lo, mwpm_ci_hi = wilson_ci(mwpm_failures, NUM_TEST)

        # ---- Transformer: pad -> reshape -> scan ----
        mask = np.ones(n_padded, dtype=np.bool_)
        p_arr = np.full(n_padded, p, dtype=np.float32)
        if pad_n:
            syndromes = np.concatenate([
                syndromes,
                np.zeros((pad_n, seq_len), dtype=np.float32)])
            labels = np.concatenate([
                labels,
                np.zeros(pad_n, dtype=labels.dtype)])
            mask[NUM_TEST:] = False
            p_arr[NUM_TEST:] = 0.0

        # Transfer + reshape for scan: (n_eval_batches, eval_batch, ...)
        syn_d  = jax.device_put(
            syndromes.reshape(n_eval_batches, eval_batch, seq_len))
        p_d    = jax.device_put(
            p_arr.reshape(n_eval_batches, eval_batch))
        lab_d  = jax.device_put(
            labels.reshape(n_eval_batches, eval_batch))
        mask_d = jax.device_put(
            mask.reshape(n_eval_batches, eval_batch))

        # One compiled call — entire eval as single XLA while-loop
        correct = int(predict_all(params, syn_d, p_d, lab_d, mask_d))
        tf_failures = NUM_TEST - correct
        tf_ler = tf_failures / NUM_TEST
        tf_ci_lo, tf_ci_hi = wilson_ci(tf_failures, NUM_TEST)

        del syn_d, p_d, lab_d, mask_d

        results[(d, p)] = {
            'mwpm_ler':   mwpm_ler,
            'mwpm_ci_lo': mwpm_ci_lo,
            'mwpm_ci_hi': mwpm_ci_hi,
            'transformer_ler': tf_ler,
            'tf_ci_lo':   tf_ci_lo,
            'tf_ci_hi':   tf_ci_hi,
        }
        print(f'd={d} p={p:.4f} | '
              f'MWPM={mwpm_ler:.6f} [{mwpm_ci_lo:.6f},{mwpm_ci_hi:.6f}] | '
              f'Transformer={tf_ler:.6f} [{tf_ci_lo:.6f},{tf_ci_hi:.6f}]')


## D₄ × T Test-Time Augmentation

Average softmax probabilities over the subgroup of D₄ × time-reversal symmetries that are valid bijective detector permutations for `rotated_memory_z`. 90°/270° rotations and diagonal reflections swap X/Z stabilizer types so they're filtered out automatically by `get_d4_permutations` — only identity, 180° rotation, and their time-reversal compositions survive (~4 elements).

In [ ]:
# --- D4 x T test-time augmentation ---
import sys
REPO_ROOT = '/content/drive/MyDrive/TransformerQEC'
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
from research_symmetries import get_d4_permutations

results_tta = {}

for d in DISTANCES:
    if d not in models:
        continue
    jax.clear_caches()
    gc.collect()

    info = models[d]
    model_d = info['model']
    params  = jax.device_put(info['params'])
    seq_len = info['seq_len']
    coords_np = np.asarray(info['coords'])

    # Build valid D4 x T permutations once; shapes are static per distance.
    perms = get_d4_permutations(coords_np, include_time_reversal=True)
    G = len(perms)
    print(f'd={d}: {G} valid D4xT permutations: {list(perms)}')
    if G == 0:
        continue

    coords_perms = jnp.stack([
        jnp.asarray(coords_np[p]) for p in perms.values()
    ])                                                       # (G, L, 3)
    perm_idx = jnp.stack([jnp.asarray(p) for p in perms.values()])  # (G, L)

    eval_batch = 128
    n_padded = ((NUM_TEST + eval_batch - 1) // eval_batch) * eval_batch
    pad_n = n_padded - NUM_TEST
    n_eval_batches = n_padded // eval_batch

    @jax.jit
    def tta_predict(params, syn_batched, p_batched, lab_batched, mask_batched):
        """Per-batch TTA: softmax-average logits over G symmetry elements,
        then argmax. Returns count of correct predictions on unmasked slots.
        """
        def body(correct, batch):
            syn, pe, lab, mask = batch                        # syn: (B, L)
            def one(coords_g, perm_g):
                return jax.nn.softmax(
                    model_d.apply({'params': params},
                                  syn[:, perm_g], pe, coords_g),
                    axis=-1)                                  # (B, 2)
            probs = jax.vmap(one)(coords_perms, perm_idx)     # (G, B, 2)
            preds = probs.mean(axis=0).argmax(-1)             # (B,)
            correct = correct + jnp.where(mask, preds == lab, False).sum()
            return correct, None
        correct, _ = jax.lax.scan(
            body, jnp.int32(0),
            (syn_batched, p_batched, lab_batched, mask_batched))
        return correct

    for p in P_EVAL:
        circuit = make_circuit(d, p)
        syndromes, labels = sample_syndromes(circuit, NUM_TEST)

        mask = np.ones(n_padded, dtype=np.bool_)
        p_arr = np.full(n_padded, p, dtype=np.float32)
        if pad_n:
            syndromes = np.concatenate([
                syndromes,
                np.zeros((pad_n, seq_len), dtype=np.float32)])
            labels = np.concatenate([
                labels,
                np.zeros(pad_n, dtype=labels.dtype)])
            mask[NUM_TEST:] = False
            p_arr[NUM_TEST:] = 0.0

        syn_d  = jax.device_put(
            syndromes.reshape(n_eval_batches, eval_batch, seq_len))
        p_d    = jax.device_put(
            p_arr.reshape(n_eval_batches, eval_batch))
        lab_d  = jax.device_put(
            labels.reshape(n_eval_batches, eval_batch))
        mask_d = jax.device_put(
            mask.reshape(n_eval_batches, eval_batch))

        correct = int(tta_predict(params, syn_d, p_d, lab_d, mask_d))
        tta_failures = NUM_TEST - correct
        tta_ler = tta_failures / NUM_TEST
        ci_lo, ci_hi = wilson_ci(tta_failures, NUM_TEST)

        del syn_d, p_d, lab_d, mask_d

        results_tta[(d, p)] = {
            'tta_ler': tta_ler,
            'ci_lo': ci_lo,
            'ci_hi': ci_hi,
            'n_group': G,
        }
        # Compare to no-TTA Transformer at same (d, p)
        base = results.get((d, p), {}).get('transformer_ler')
        delta = (base - tta_ler) / base * 100 if base and base > 0 else float('nan')
        print(f'd={d} p={p:.4f} | TTA={tta_ler:.6f} [{ci_lo:.6f},{ci_hi:.6f}] '
              f'(G={G}, delta vs no-TTA: {delta:+.1f}%)')


## Logical Error Rate Curves

The key diagnostic: $p_L$ vs $p$ for each decoder and distance. Curves for different $d$ should cross near the threshold $p_{\text{th}}$.

In [ ]:
SAVE_DIR = CKPT_DIR  # same Google Drive folder used by notebook 02

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for ax, (ler_key, lo_key, hi_key, title) in zip(axes, [
    ('mwpm_ler',        'mwpm_ci_lo', 'mwpm_ci_hi', 'MWPM Decoder'),
    ('transformer_ler', 'tf_ci_lo',   'tf_ci_hi',   'Transformer Decoder')]):

    for d in DISTANCES:
        ps   = [p for p in P_EVAL if (d, p) in results]
        lers = [results[(d, p)][ler_key] for p in ps]
        los  = [results[(d, p)][lo_key]  for p in ps]
        his  = [results[(d, p)][hi_key]  for p in ps]
        # Filter zeros for log scale (keep CI band on valid points)
        keep = [l > 0 for l in lers]
        ps_k  = [pi for pi, k in zip(ps,   keep) if k]
        le_k  = [li for li, k in zip(lers, keep) if k]
        lo_k  = [max(li, 1e-12) for li, k in zip(los, keep) if k]
        hi_k  = [hi for hi, k in zip(his,  keep) if k]
        line, = ax.plot(ps_k, le_k, 'o-', label=f'd={d}', markersize=5)
        ax.fill_between(ps_k, lo_k, hi_k,
                        alpha=0.2, color=line.get_color(), linewidth=0)

    ax.set(xlabel='Physical error rate (p)',
           ylabel='Logical error rate',
           xscale='log', yscale='log', title=title)
    ax.legend(); ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
fig.savefig(SAVE_DIR / 'logical_error_rates.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR / "logical_error_rates.png"}')
plt.show()

In [ ]:
# Combined comparison: MWPM (dashed) vs Transformer (solid), both with 95% CI bands.
fig, ax = plt.subplots(figsize=(15, 6))
colors = {3: 'C0', 5: 'C1', 7: 'C2'}


def nonzero_triples(ps, vals, los, his):
    """Keep (p, val, lo, hi) only where val > 0 (log-scale safety)."""
    out = [(p, v, l, h) for p, v, l, h in zip(ps, vals, los, his) if v > 0]
    if not out:
        return [], [], [], []
    ps_k, vs_k, ls_k, hs_k = zip(*out)
    return list(ps_k), list(vs_k), [max(l, 1e-12) for l in ls_k], list(hs_k)


for d in DISTANCES:
    ps   = [p for p in P_EVAL if (d, p) in results]
    mwpm = [results[(d, p)]['mwpm_ler'] for p in ps]
    mlo  = [results[(d, p)]['mwpm_ci_lo'] for p in ps]
    mhi  = [results[(d, p)]['mwpm_ci_hi'] for p in ps]
    tf   = [results[(d, p)]['transformer_ler'] for p in ps]
    tlo  = [results[(d, p)]['tf_ci_lo'] for p in ps]
    thi  = [results[(d, p)]['tf_ci_hi'] for p in ps]

    pm, lm, mlo_k, mhi_k = nonzero_triples(ps, mwpm, mlo, mhi)
    if pm:
        ax.plot(pm, lm, 's--', color=colors[d], alpha=0.7,
                label=f'MWPM d={d}')
        ax.fill_between(pm, mlo_k, mhi_k,
                        alpha=0.12, color=colors[d], linewidth=0)
    pt, lt, tlo_k, thi_k = nonzero_triples(ps, tf, tlo, thi)
    if pt:
        ax.plot(pt, lt, 'o-', color=colors[d],
                label=f'Transformer d={d}')
        ax.fill_between(pt, tlo_k, thi_k,
                        alpha=0.2, color=colors[d], linewidth=0)

# Overlay TTA curve if cell-12b has run
if 'results_tta' in globals():
    for d in DISTANCES:
        ps = [p for p in P_EVAL if (d, p) in results_tta]
        if not ps:
            continue
        tta  = [results_tta[(d, p)]['tta_ler'] for p in ps]
        tlo  = [results_tta[(d, p)]['ci_lo']  for p in ps]
        thi  = [results_tta[(d, p)]['ci_hi']  for p in ps]
        pk, vk, lk, hk = nonzero_triples(ps, tta, tlo, thi)
        if pk:
            ax.plot(pk, vk, '^:', color=colors[d], alpha=0.9,
                    label=f'Transformer+TTA d={d}')
            ax.fill_between(pk, lk, hk,
                            alpha=0.15, color=colors[d], linewidth=0)

ax.set(xlabel='Physical error rate (p)',
       ylabel='Logical error rate',
       xscale='log', yscale='log',
       title='Transformer vs MWPM: Logical Error Rate (shaded = 95% Wilson CI)')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
fig.savefig(SAVE_DIR / 'transformer_vs_mwpm.png', dpi=150, bbox_inches='tight')
print(f'Saved: {SAVE_DIR / "transformer_vs_mwpm.png"}')
plt.show()

## Comparison Table

In [ ]:
import csv

# Print comparison table
header = f'{"d":>3} {"p":>8} {"MWPM":>12} {"Transformer":>12} {"Improvement":>12}'
print(header)
print('-' * 53)

rows = []
for d in DISTANCES:
    for p in P_EVAL:
        if (d, p) not in results:
            continue
        r = results[(d, p)]
        m, t = r['mwpm_ler'], r['transformer_ler']
        imp = (m - t) / m * 100 if m > 0 else float('nan')
        rows.append({
            'd': d, 'p': p,
            'mwpm_ler': m,
            'mwpm_ci_lo': r['mwpm_ci_lo'],
            'mwpm_ci_hi': r['mwpm_ci_hi'],
            'transformer_ler': t,
            'tf_ci_lo': r['tf_ci_lo'],
            'tf_ci_hi': r['tf_ci_hi'],
            'improvement_pct': imp,
        })
        if m > 0:
            print(f'{d:>3} {p:>8.4f} {m:>12.6f} {t:>12.6f} {imp:>+11.1f}%')
        elif t > 0:
            print(f'{d:>3} {p:>8.4f} {m:>12.6f} {t:>12.6f} {"(MWPM=0)":>12}')
        else:
            print(f'{d:>3} {p:>8.4f} {m:>12.6f} {t:>12.6f} {"both 0":>12}')

# Save results to CSV
csv_path = SAVE_DIR / 'evaluation_results.csv'
fieldnames = ['d', 'p',
              'mwpm_ler', 'mwpm_ci_lo', 'mwpm_ci_hi',
              'transformer_ler', 'tf_ci_lo', 'tf_ci_hi',
              'improvement_pct']
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)
print(f'\nSaved: {csv_path}')

## Threshold Estimation

The threshold $p_{\text{th}}$ is where logical error rate curves for different distances cross. Below threshold, higher $d$ gives lower $p_L$; above threshold, higher $d$ gives higher $p_L$.

In [ ]:
def estimate_threshold(d1, d2, decoder_key):
    """Find crossing point of p_L curves for two distances."""
    ps, ler1, ler2 = [], [], []
    for p in P_EVAL:
        if (d1, p) in results and (d2, p) in results:
            l1 = results[(d1, p)][decoder_key]
            l2 = results[(d2, p)][decoder_key]
            if l1 > 0 and l2 > 0:
                ps.append(p); ler1.append(l1); ler2.append(l2)
    if len(ps) < 2:
        return None
    log_ps = np.log(ps)
    diff = np.log(ler2) - np.log(ler1)
    for i in range(len(diff) - 1):
        if diff[i] * diff[i + 1] < 0:  # sign change
            t = diff[i] / (diff[i] - diff[i + 1])
            return np.exp(log_ps[i] + t * (log_ps[i + 1] - log_ps[i]))
    return None


# Build distance pairs from loaded models so we don't reference unavailable distances
distance_pairs = list(zip(DISTANCES[:-1], DISTANCES[1:]))

threshold_lines = []
threshold_lines.append('Threshold estimates (from curve crossings):')
threshold_lines.append('=' * 55)
for key, name in [('mwpm_ler', 'MWPM'), ('transformer_ler', 'Transformer')]:
    for d1, d2 in distance_pairs:
        p_th = estimate_threshold(d1, d2, key)
        if p_th:
            threshold_lines.append(f'{name:>12} d={d1}/{d2} crossing: '
                  f'p_th ~ {p_th:.4f}')
        else:
            threshold_lines.append(f'{name:>12} d={d1}/{d2} crossing: '
                  f'not found in evaluated range')

for line in threshold_lines:
    print(line)

# Save threshold estimates
txt_path = SAVE_DIR / 'threshold_estimates.txt'
with open(txt_path, 'w') as f:
    f.write('\n'.join(threshold_lines) + '\n')
print(f'\nSaved: {txt_path}')

In [25]:
# Evaluation outputs (plots, CSV, threshold estimates) are saved
# directly to Google Drive at CKPT_DIR. No extraction step needed.

## Summary

**Key findings:**
- The transformer decoder learns to predict logical errors from noisy syndrome patterns.
- At moderate error rates ($p \sim 0.02$-$0.08$), the transformer can match or exceed MWPM by capturing Y-error correlations that MWPM's independent X/Z decoding misses.
- Threshold estimates from both decoders should be comparable ($p_{\text{th}} \approx 0.09$-$0.10$ for phenomenological noise).

**Limitations & extensions:**
- Phenomenological noise only (not circuit-level).
- Binary classification (logical Z only, not full 4-class I/X/Z/Y).
- Small distances ($d \leq 7$). Distance generalization ($d=9$ without retraining) is a natural next step.
- Attention map analysis could reveal whether the model learns matching-like pairwise defect correlations.
- Circuit-level noise via Stim's full circuit simulation would bring this closer to AlphaQubit's setup.